# 🐍 Day 1: Threading

- Thread
- Lock, RLock
- Event, Condition
- threading module

## Why Threads?

Threads run code concurrently within one process. Good for I/O-bound work (network, files). Python's GIL limits CPU parallelism.

## Creating Threads

Use `threading.Thread` with a target function or by subclassing.

In [ ]:
import threading
import time

def worker(name):
    print(f"{name} starting")
    time.sleep(1)
    print(f"{name} done")

t1 = threading.Thread(target=worker, args=("Thread-1",))
t2 = threading.Thread(target=worker, args=("Thread-2",))

t1.start()
t2.start()
t1.join()
t2.join()
print("All done")

## Lock (Mutual Exclusion)

Protect shared data from race conditions. Only one thread holds the lock at a time.

In [ ]:
import threading

counter = 0
lock = threading.Lock()

def increment():
    global counter
    for _ in range(100000):
        with lock:
            counter += 1

threads = [threading.Thread(target=increment) for _ in range(4)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"Counter: {counter}")

## RLock (Reentrant Lock)

Same thread can acquire the lock multiple times. Needed when a function that holds the lock calls another that also needs it.

In [ ]:
import threading

rlock = threading.RLock()

def outer():
    with rlock:
        print("outer")
        inner()

def inner():
    with rlock:  # Same thread, so OK with RLock
        print("inner")

outer()

## Event

One thread signals others. `set()` wakes waiters; `clear()` resets; `wait()` blocks until set.

In [ ]:
import threading
import time

event = threading.Event()

def waiter():
    print("Waiting for event...")
    event.wait()
    print("Event received!")

def setter():
    time.sleep(2)
    print("Setting event")
    event.set()

t1 = threading.Thread(target=waiter)
t2 = threading.Thread(target=setter)
t1.start()
t2.start()
t1.join()
t2.join()

## Condition

Wait for a condition (e.g., queue not empty). Combines Lock with wait/notify.

In [ ]:
import threading
import time

condition = threading.Condition()
items = []

def consumer():
    with condition:
        condition.wait_for(lambda: len(items) > 0)
        item = items.pop()
        print(f"Consumed {item}")

def producer():
    time.sleep(1)
    with condition:
        items.append(42)
        condition.notify()

t1 = threading.Thread(target=consumer)
t2 = threading.Thread(target=producer)
t1.start()
t2.start()
t1.join()
t2.join()

## Daemon Threads

Daemon threads don't block process exit. Use for background tasks.

In [ ]:
import threading
import time

def background():
    while True:
        print(".", end="", flush=True)
        time.sleep(0.5)

t = threading.Thread(target=background, daemon=True)
t.start()
time.sleep(2)
print("\nMain done - daemon exits with main")

## Common Mistakes

- **Forgetting to acquire lock**: Shared data can corrupt
- **Deadlock**: Lock A then B in one thread, B then A in another
- **Using Lock instead of RLock**: Recursive lock acquisition fails
- **CPU-bound work in threads**: GIL limits; use multiprocessing

## Exercises

In [ ]:
# Exercise 1: Create two threads that append to a shared list. Use a Lock to avoid races.
# YOUR CODE HERE

In [ ]:
# Exercise 2: Use Event to make a thread wait until another sets a "ready" signal.
# YOUR CODE HERE

In [ ]:
# Exercise 3: Implement a simple thread-safe counter class with increment() and value.
# YOUR CODE HERE

In [ ]:
# Exercise 4: Use Condition to implement a bounded buffer (1 slot). Producer waits if full.
# YOUR CODE HERE

In [ ]:
# Exercise 5: Create a daemon thread that logs every second. Let main sleep 3 seconds then exit.
# YOUR CODE HERE

In [ ]:
# Exercise 6: Use threading.current_thread().name to print which thread is running.
# YOUR CODE HERE

## Key Takeaways

- Thread for I/O-bound concurrency
- Lock for mutual exclusion; RLock for reentrant
- Event for simple signaling; Condition for wait/notify
- Daemon threads exit with main

## 🔜 Next: Day 2 - Thread Pools

Tomorrow: ThreadPoolExecutor, as_completed, map!